# 🌏 Tour Content Rewriter — Agentic AI System

**Domain:** Rewrite tour content from Excel → SQL INSERT statements (matching DB schema)

## Architecture Overview
```
Excel (Tiger Trail Products)
        │
        ▼
┌───────────────────────────────────────────────┐
│              LangGraph Orchestrator            │
│                                               │
│  parse_node → rewrite_node → map_node         │
│      │            │             │             │
│  [Guardrail]  [LLM+Cache]  [SQL Builder]      │
│      │            │             │             │
│  validator_node ← ← ← ← ← ← ← ┘             │
│      │                                        │
│  [Retry on fail / Fallback]                   │
└───────────────────────────────────────────────┘
        │
        ▼
   SQL INSERT statements
```

## Scoring Targets (from Criteria sheet)
| Criterion | Target | Strategy |
|-----------|--------|----------|
| Problem & System Design | 9-10 | Clear workflow, justified architecture |
| Agent Architecture | 13-15 | LangGraph + memory + state handling |
| Tooling & Integration | 9-10 | Excel reader, SQL builder, schema validator |
| Retrieval/RAG | 9-10 | Schema as knowledge base + semantic matching |
| Observability | 9-10 | Langfuse spans + token tracking |
| Evaluation | 13-15 | Automated eval + regression dataset |
| Reliability & Guardrails | 9-10 | Pydantic validation + fallback |
| Performance & Cost | 5 | Caching + batching |
| Production Readiness | 9-10 | Modular, config-driven |
| Demo & Explanation | 5 | Live demo with metrics |

## Setup

In [ ]:
import os
import json
import hashlib
import logging
import re
import time
from datetime import datetime
from typing import TypedDict, Optional, Annotated
from pathlib import Path

import openpyxl
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("tour_rewriter")

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

# ── LLM Configuration ────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.3,  # Some creativity for rewrites
    max_tokens=2000,
)

# Cheaper LLM for validation (cost optimization)
llm_fast = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0,
    max_tokens=500,
)

logger.info("Setup complete")

## Criterion 5: Observability — Langfuse Tracing

In [ ]:
# ── Observability: Langfuse (graceful fallback if not configured) ────────────
try:
    from langfuse import Langfuse
    from langfuse.decorators import observe, langfuse_context

    if LANGFUSE_SECRET_KEY and LANGFUSE_PUBLIC_KEY:
        langfuse = Langfuse(
            secret_key=LANGFUSE_SECRET_KEY,
            public_key=LANGFUSE_PUBLIC_KEY,
            host=LANGFUSE_HOST,
        )
        OBSERVABILITY_ENABLED = True
        logger.info("✓ Langfuse observability enabled")
    else:
        OBSERVABILITY_ENABLED = False
        logger.warning("Langfuse keys not found — observability disabled")
except ImportError:
    OBSERVABILITY_ENABLED = False
    logger.warning("langfuse not installed — observability disabled")


class ObservabilityTracker:
    """Tracks spans, token usage, and costs for each pipeline run."""

    def __init__(self):
        self.spans = []
        self.total_tokens = 0
        self.total_cost = 0.0
        self.start_time = time.time()

    def log_span(self, name: str, input_data: dict, output_data: dict,
                 tokens: int = 0, latency_ms: int = 0, metadata: dict = None):
        span = {
            "name": name,
            "timestamp": datetime.utcnow().isoformat(),
            "input": str(input_data)[:500],
            "output": str(output_data)[:500],
            "tokens": tokens,
            "latency_ms": latency_ms,
            "metadata": metadata or {},
        }
        self.spans.append(span)
        self.total_tokens += tokens
        # GPT-4o-mini: ~$0.15/1M input, ~$0.60/1M output (rough avg $0.40/1M)
        self.total_cost += tokens * 0.0000004
        logger.info(f"  [SPAN] {name} | tokens={tokens} | latency={latency_ms}ms")

    def summary(self) -> dict:
        total_time = (time.time() - self.start_time) * 1000
        return {
            "total_spans": len(self.spans),
            "total_tokens": self.total_tokens,
            "estimated_cost_usd": round(self.total_cost, 4),
            "total_latency_ms": round(total_time, 1),
            "spans": self.spans,
        }


tracker = ObservabilityTracker()
logger.info("✓ ObservabilityTracker initialized")

## Criterion 7: Reliability & Guardrails — Pydantic Models + Validation

In [ ]:
# ── Pydantic Models for data validation (Guardrails) ─────────────────────────

class RawTourProduct(BaseModel):
    """Raw data from Excel — loose validation."""
    sheet_name: str
    sku: Optional[str] = None
    name: str
    subtitle: Optional[str] = None
    duration: Optional[str] = None
    itineraries_raw: Optional[str] = None  # Full text of all days
    summary: Optional[str] = None
    highlights: Optional[str] = None
    provider: Optional[str] = None
    price_raw: Optional[str] = None
    inclusions: Optional[str] = None
    exclusions: Optional[str] = None
    description: Optional[str] = None

    @field_validator('name')
    @classmethod
    def name_not_empty(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("Tour name cannot be empty")
        return v[:255]  # DB limit


class RewrittenContent(BaseModel):
    """LLM-rewritten content — strict validation."""
    title: str = Field(min_length=5, max_length=255)
    summary: str = Field(min_length=20, max_length=2000)
    highlight: str = Field(min_length=20, max_length=5000)
    included: str = Field(min_length=10, max_length=3000)
    not_included: str = Field(min_length=10, max_length=3000)
    duration: Optional[str] = Field(default=None, max_length=100)
    sku: Optional[str] = Field(default=None, max_length=100)

    @field_validator('title')
    @classmethod
    def no_sql_injection(cls, v: str) -> str:
        dangerous = ["DROP", "DELETE", "INSERT", "UPDATE", "EXEC", "--", "'"]
        for d in dangerous:
            if d.upper() in v.upper():
                raise ValueError(f"Potential SQL injection detected: {d}")
        return v.replace("'", "''")  # Escape single quotes for SQL


class ItineraryDay(BaseModel):
    """A single day in the tour itinerary."""
    day_number: int = Field(ge=1)
    title: str = Field(min_length=3, max_length=255)
    description: str = Field(min_length=10, max_length=10000)
    level: Optional[str] = Field(default="moderate", max_length=100)


class SQLOutput(BaseModel):
    """Validated SQL INSERT statements."""
    provider_sql: Optional[str] = None
    destination_sql: Optional[str] = None
    tour_sql: str
    itinerary_sqls: list[str]
    tour_name: str
    validation_passed: bool = False
    validation_errors: list[str] = Field(default_factory=list)


logger.info("✓ Pydantic guardrail models defined")

## Criterion 8: Performance & Cost — Caching Layer

In [ ]:
# ── Caching: avoid redundant LLM calls ───────────────────────────────────────

class LLMCache:
    """Simple file-based cache for LLM responses to reduce cost."""

    def __init__(self, cache_file: str = "/home/claude/llm_cache.json"):
        self.cache_file = cache_file
        self._cache = self._load()
        self.hits = 0
        self.misses = 0

    def _load(self) -> dict:
        try:
            if Path(self.cache_file).exists():
                with open(self.cache_file) as f:
                    return json.load(f)
        except Exception:
            pass
        return {}

    def _save(self):
        try:
            with open(self.cache_file, "w") as f:
                json.dump(self._cache, f, ensure_ascii=False, indent=2)
        except Exception as e:
            logger.warning(f"Cache save failed: {e}")

    def _key(self, text: str) -> str:
        return hashlib.md5(text.encode()).hexdigest()

    def get(self, prompt: str) -> Optional[str]:
        key = self._key(prompt)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        self.misses += 1
        return None

    def set(self, prompt: str, response: str):
        key = self._key(prompt)
        self._cache[key] = response
        self._save()

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{self.hits/total*100:.1f}%" if total > 0 else "0%",
            "cached_entries": len(self._cache),
        }


cache = LLMCache()
logger.info(f"✓ LLM cache initialized ({len(cache._cache)} cached entries)")

## Criterion 4: Retrieval/RAG — Schema as Knowledge Base

In [ ]:
# ── Schema RAG: embed DB schema so LLM understands target structure ───────────

DB_SCHEMA_KNOWLEDGE = """
DATABASE SCHEMA KNOWLEDGE BASE:

TABLE: providers
- id (BIGINT AUTO_INCREMENT PK)
- name VARCHAR(255) NOT NULL  — e.g. "Tiger Trail", "Tiger Travel"
- contact_name VARCHAR(255)
- phone VARCHAR(50)
- email VARCHAR(255)

TABLE: areas
- id (BIGINT AUTO_INCREMENT PK)
- name VARCHAR(255) NOT NULL  — e.g. "Southeast Asia", "Laos", "Indochina"

TABLE: countries
- id (BIGINT AUTO_INCREMENT PK)
- area_id FK → areas
- name VARCHAR(255) NOT NULL  — e.g. "Laos", "Vietnam"
- subtitle TEXT  — short tagline for the country
- description LONGTEXT  — marketing description
- status ENUM('ACTIVE','INACTIVE') DEFAULT 'ACTIVE'

TABLE: destinations
- id (BIGINT AUTO_INCREMENT PK)
- country_id FK → countries
- name VARCHAR(255) NOT NULL  — e.g. "Luang Prabang", "Vientiane", "Pakse"
- status ENUM('ACTIVE','INACTIVE') DEFAULT 'ACTIVE'

TABLE: tours  ← PRIMARY TARGET
- id (BIGINT AUTO_INCREMENT PK)
- destination_id FK → destinations
- sku VARCHAR(100)  — unique tour code
- title VARCHAR(255) NOT NULL  — compelling marketing title
- duration VARCHAR(100)  — e.g. "4 Days 3 Nights", "2 Days"
- price DECIMAL(10,2)  — base price in USD
- summary TEXT  — 2-3 sentence hook paragraph
- hightlight LONGTEXT  — bullet points of what makes it special
- included LONGTEXT  — what's included in the price
- not_included LONGTEXT  — what's NOT included (exclusions)
- status ENUM('ACTIVE','INACTIVE') DEFAULT 'ACTIVE'

TABLE: itineraries
- id (BIGINT AUTO_INCREMENT PK)
- tour_id FK → tours
- title VARCHAR(255) NOT NULL  — day title, e.g. "Day 1: Arrival in Luang Prabang"
- level VARCHAR(100)  — difficulty: 'easy', 'moderate', 'challenging'
- description LONGTEXT  — full day description (rewritten, engaging)
- status ENUM('ACTIVE','INACTIVE') DEFAULT 'ACTIVE'

MAPPING RULES:
- tours.summary → 2-3 engaging sentences, hook the reader
- tours.hightlight → bullet points starting with emoji or •
- tours.included → clean bullet list of inclusions
- tours.not_included → clean bullet list of exclusions
- itineraries.title → "Day X: [Location] — [Activity Theme]"
- itineraries.level → infer from activity: hiking=challenging, city tour=easy, mix=moderate
- itineraries.description → rewrite as engaging narrative, preserve all facts
"""

# Schema lookup function (simulates vector retrieval for relevant schema parts)
def retrieve_schema_context(field_name: str) -> str:
    """RAG: retrieve relevant schema knowledge for a given field."""
    field_map = {
        "summary": "tours.summary: 2-3 sentence engaging hook. Mention key destinations and unique experiences.",
        "highlight": "tours.hightlight: Bullet points with emojis. Each bullet = one unique selling point.",
        "included": "tours.included: Clean list. Start each item with '✓'. Factual, no fluff.",
        "not_included": "tours.not_included: Clean list. Start each item with '✗'. Be specific.",
        "itinerary": "itineraries: title must be 'Day N: Location — Theme'. description=engaging narrative.",
        "title": "tours.title: Max 255 chars. Should be evocative and include key destination.",
    }
    return field_map.get(field_name, DB_SCHEMA_KNOWLEDGE)


logger.info("✓ Schema RAG knowledge base loaded")

## Criterion 3: Tooling — Excel Parser Tool

In [ ]:
# ── Tool 1: Excel Parser ──────────────────────────────────────────────────────

def parse_excel_tours(excel_path: str) -> list[RawTourProduct]:
    """
    Tool: Parse Tiger Trail Excel into structured RawTourProduct objects.
    Handles the multi-row format where itinerary days are spread across rows.
    """
    wb = openpyxl.load_workbook(excel_path, read_only=True)
    tours = []

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = list(ws.iter_rows(values_only=True))

        # Find header row
        header_row = None
        for i, row in enumerate(rows[:3]):
            if any(str(cell).upper() in ['SKU', 'NAME', 'DURATION'] for cell in row if cell):
                header_row = i
                break

        if header_row is None:
            continue

        headers = [str(h).upper().strip() if h else '' for h in rows[header_row]]

        def get_col(row, name):
            for i, h in enumerate(headers):
                if name in h and i < len(row):
                    return str(row[i]).strip() if row[i] else None
            return None

        # Group rows by tour (a tour starts when NAME column has a value)
        current_tour = None
        itinerary_days = []
        inclusions_found = []
        exclusions_found = []
        reading_inclusions = False
        reading_exclusions = False

        for row in rows[header_row + 2:]:  # skip header + INCLUSIONS sub-header
            if not any(cell for cell in row):
                continue

            name_val = None
            for i, h in enumerate(headers):
                if 'NAME' in h and i < len(row) and row[i]:
                    name_val = str(row[i]).strip()
                    break

            itin_val = get_col(row, 'ITINERAR')
            incl_val = get_col(row, 'INCLUS')
            excl_val = get_col(row, 'EXCLUS')

            if incl_val:
                inclusions_found.append(incl_val)
            if excl_val:
                exclusions_found.append(excl_val)

            if name_val and name_val not in ["None", ""]:
                # Save previous tour
                if current_tour is not None:
                    current_tour.itineraries_raw = "\n\n".join(itinerary_days)
                    if inclusions_found:
                        current_tour.inclusions = inclusions_found[0] if inclusions_found else None
                    if exclusions_found:
                        current_tour.exclusions = exclusions_found[0] if exclusions_found else None
                    if len(name_val) >= 3:  # Basic sanity check
                        try:
                            tours.append(current_tour)
                        except Exception as e:
                            logger.warning(f"Skipping tour due to validation error: {e}")

                # Start new tour
                itinerary_days = []
                inclusions_found = []
                exclusions_found = []

                try:
                    current_tour = RawTourProduct(
                        sheet_name=sheet_name,
                        sku=get_col(row, 'SKU'),
                        name=name_val[:255],
                        subtitle=get_col(row, 'SUBTITLE'),
                        duration=get_col(row, 'DURATION'),
                        summary=get_col(row, 'SUMMARY'),
                        highlights=get_col(row, 'HIGHLIGHT'),
                        provider=get_col(row, 'PROVIDER'),
                        price_raw=get_col(row, 'PRICE'),
                        description=get_col(row, 'DESCRIPTION'),
                    )
                    if itin_val:
                        itinerary_days.append(itin_val)
                except Exception as e:
                    logger.warning(f"Skipping row in {sheet_name}: {e}")
                    current_tour = None

            elif current_tour is not None and itin_val:
                itinerary_days.append(itin_val)

        # Don't forget the last tour
        if current_tour is not None:
            current_tour.itineraries_raw = "\n\n".join(itinerary_days)
            if inclusions_found:
                current_tour.inclusions = inclusions_found[0]
            if exclusions_found:
                current_tour.exclusions = exclusions_found[0]
            try:
                tours.append(current_tour)
            except Exception:
                pass

    logger.info(f"✓ Parsed {len(tours)} tours from {len(wb.sheetnames)} sheets")
    return tours


# Test parse
raw_tours = parse_excel_tours("/mnt/project/tiger_trail_s_products.xlsx")
logger.info(f"Sample: {raw_tours[0].name[:60]} | Sheet: {raw_tours[0].sheet_name}")

## Criterion 2: Agent Architecture — LangGraph State & Nodes

In [ ]:
# ── LangGraph State Definition ────────────────────────────────────────────────

class TourProcessingState(TypedDict):
    """Full state flowing through the graph for ONE tour."""
    # Input
    raw_tour: dict                    # Serialized RawTourProduct
    tour_index: int

    # Processing state
    rewritten_content: Optional[dict]  # Serialized RewrittenContent
    itinerary_days: list[dict]         # List of ItineraryDay
    sql_output: Optional[dict]         # Serialized SQLOutput

    # Orchestration
    validation_passed: bool
    retry_count: int
    errors: list[str]
    warnings: list[str]

    # Observability
    trace_id: str
    node_timings: dict


# ── System Prompts (managed, versionable) ─────────────────────────────────────

REWRITE_SYSTEM_PROMPT = """You are an expert travel copywriter for Tiger Trail, 
a premium tour operator specializing in authentic Laos experiences.

Your task: Transform raw tour data into polished, engaging marketing content.

STYLE GUIDELINES:
- Tone: Adventurous yet trustworthy, evocative but factual
- Voice: Second person ("you will discover", "your guide will")
- Length: Summary 2-3 sentences; Highlights 5-8 bullet points
- Always preserve: dates, prices, locations, durations, specific site names
- Never fabricate facts; only enhance language and structure

SCHEMA CONTEXT:
{schema_context}

Return ONLY valid JSON matching exactly this structure:
{{
  "title": "...",
  "summary": "...",
  "highlight": "...",
  "included": "...",
  "not_included": "...",
  "duration": "...",
  "sku": "..."
}}
"""

ITINERARY_SYSTEM_PROMPT = """You are a travel content specialist.
Parse the raw itinerary text and rewrite each day as an engaging narrative.

For each day return JSON array:
[{{
  "day_number": 1,
  "title": "Day 1: Luang Prabang — Ancient Temples & Golden Monks",
  "description": "Engaging rewrite of the day's activities...",
  "level": "easy|moderate|challenging"
}}]

Rules:
- Preserve ALL factual information (temple names, activities, distances)
- Infer difficulty from activities (city tour=easy, 10km trek=challenging)
- title format: "Day N: [Main Location] — [Theme]"
- Return ONLY the JSON array, no other text
"""

logger.info("✓ LangGraph state and prompts defined")

In [ ]:
# ── Node 1: Parse & Validate Input ────────────────────────────────────────────

def parse_node(state: TourProcessingState) -> TourProcessingState:
    """Validate input, extract price, normalize provider name."""
    start = time.time()
    raw = state["raw_tour"]
    errors = list(state.get("errors", []))
    warnings = list(state.get("warnings", []))

    # Guardrail: validate required fields
    if not raw.get("name"):
        errors.append("CRITICAL: Tour name is empty")
        return {**state, "errors": errors, "validation_passed": False}

    # Normalize: extract first numeric price
    price_raw = raw.get("price_raw", "")
    price = None
    if price_raw:
        nums = re.findall(r'[\d,]+\.?\d*', str(price_raw).replace(',', ''))
        if nums:
            try:
                price = float(nums[0])
            except ValueError:
                warnings.append(f"Could not parse price: {price_raw[:50]}")

    raw["price_parsed"] = price

    # Normalize provider
    provider = (raw.get("provider") or "Tiger Trail").strip().title()
    raw["provider_normalized"] = provider

    ms = int((time.time() - start) * 1000)
    timings = dict(state.get("node_timings", {}))
    timings["parse_node"] = ms

    tracker.log_span("parse_node", {"name": raw.get("name")},
                     {"price": price, "provider": provider}, latency_ms=ms)

    return {**state, "raw_tour": raw, "errors": errors,
            "warnings": warnings, "node_timings": timings}


# ── Node 2: Rewrite Content with LLM (+ Cache) ─────────────────────────────────

def rewrite_node(state: TourProcessingState) -> TourProcessingState:
    """Use LLM to rewrite tour content into polished marketing copy."""
    start = time.time()
    raw = state["raw_tour"]
    errors = list(state.get("errors", []))

    schema_ctx = "\n".join([
        retrieve_schema_context("summary"),
        retrieve_schema_context("highlight"),
        retrieve_schema_context("included"),
    ])

    user_content = f"""
Tour Name: {raw.get('name', '')}
Subtitle: {raw.get('subtitle', '')}
Duration: {raw.get('duration', '')}
Sheet/Category: {raw.get('sheet_name', '')}
Summary: {(raw.get('summary') or '')[:500]}
Highlights: {(raw.get('highlights') or '')[:500]}
Inclusions: {(raw.get('inclusions') or '')[:400]}
Exclusions: {(raw.get('exclusions') or '')[:400]}
Description: {(raw.get('description') or '')[:400]}
SKU: {raw.get('sku') or 'N/A'}
"""

    # Cache check
    cache_key = user_content
    cached = cache.get(cache_key)

    if cached:
        raw_json = cached
        tokens_used = 0
    else:
        try:
            messages = [
                SystemMessage(content=REWRITE_SYSTEM_PROMPT.format(schema_context=schema_ctx)),
                HumanMessage(content=user_content),
            ]
            response = llm.invoke(messages)
            raw_json = response.content
            tokens_used = response.response_metadata.get('token_usage', {}).get('total_tokens', 0)
            cache.set(cache_key, raw_json)
        except Exception as e:
            errors.append(f"Rewrite LLM error: {str(e)[:100]}")
            # Fallback: use original data
            raw_json = json.dumps({
                "title": raw.get('name', 'Tour'),
                "summary": raw.get('summary') or f"Explore the wonders of {raw.get('name', 'Laos')}.",
                "highlight": raw.get('highlights') or "• Authentic cultural experience\n• Expert local guide",
                "included": raw.get('inclusions') or "• Transportation\n• Guide",
                "not_included": raw.get('exclusions') or "• International flights\n• Travel insurance",
                "duration": raw.get('duration'),
                "sku": raw.get('sku'),
            })
            tokens_used = 0

    # Parse JSON (strip markdown if needed)
    cleaned = raw_json.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```[a-z]*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```$', '', cleaned)

    try:
        parsed = json.loads(cleaned)
        # Guardrail: validate with Pydantic
        rewritten = RewrittenContent(
            title=parsed.get('title', raw.get('name', ''))[:255],
            summary=parsed.get('summary', '')[:2000] or 'Tour description coming soon.',
            highlight=parsed.get('highlight', '')[:5000] or '• Authentic experience',
            included=parsed.get('included', '')[:3000] or '• Transportation\n• Guide',
            not_included=parsed.get('not_included', '')[:3000] or '• Flights',
            duration=parsed.get('duration', raw.get('duration')),
            sku=parsed.get('sku', raw.get('sku')),
        )
    except Exception as e:
        errors.append(f"Rewrite parsing error: {str(e)[:100]}")
        rewritten = RewrittenContent(
            title=raw.get('name', 'Tour')[:255],
            summary=raw.get('summary') or 'Discover the beauty of Laos.',
            highlight=raw.get('highlights') or '• Authentic experience',
            included=raw.get('inclusions') or '• Guide included',
            not_included=raw.get('exclusions') or '• Flights not included',
        )

    ms = int((time.time() - start) * 1000)
    timings = dict(state.get("node_timings", {}))
    timings["rewrite_node"] = ms

    tracker.log_span("rewrite_node", {"tour": raw.get('name', '')[:50]},
                     {"title": rewritten.title[:50]},
                     tokens=tokens_used, latency_ms=ms,
                     metadata={"cached": cached is not None})

    return {**state, "rewritten_content": rewritten.model_dump(),
            "errors": errors, "node_timings": timings}


# ── Node 3: Parse Itineraries ─────────────────────────────────────────────────

def itinerary_node(state: TourProcessingState) -> TourProcessingState:
    """Parse and rewrite daily itineraries."""
    start = time.time()
    raw = state["raw_tour"]
    errors = list(state.get("errors", []))
    itin_raw = raw.get("itineraries_raw", "")

    if not itin_raw or len(itin_raw.strip()) < 20:
        return {**state, "itinerary_days": [],
                "warnings": state.get("warnings", []) + ["No itinerary data found"]}

    prompt_text = f"Raw itinerary text:\n\n{itin_raw[:3000]}"
    cached = cache.get(prompt_text)

    if cached:
        raw_json = cached
        tokens_used = 0
    else:
        try:
            messages = [
                SystemMessage(content=ITINERARY_SYSTEM_PROMPT),
                HumanMessage(content=prompt_text),
            ]
            response = llm.invoke(messages)
            raw_json = response.content
            tokens_used = response.response_metadata.get('token_usage', {}).get('total_tokens', 0)
            cache.set(prompt_text, raw_json)
        except Exception as e:
            errors.append(f"Itinerary LLM error: {str(e)[:100]}")
            raw_json = "[]"
            tokens_used = 0

    # Parse
    cleaned = raw_json.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```[a-z]*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```$', '', cleaned)

    try:
        days_raw = json.loads(cleaned)
        if not isinstance(days_raw, list):
            days_raw = []

        validated_days = []
        for day in days_raw:
            try:
                validated = ItineraryDay(
                    day_number=int(day.get('day_number', 1)),
                    title=str(day.get('title', f"Day {day.get('day_number', 1)}"))[:255],
                    description=str(day.get('description', ''))[:10000],
                    level=str(day.get('level', 'moderate'))[:100],
                )
                validated_days.append(validated.model_dump())
            except Exception as e:
                errors.append(f"Day validation error: {str(e)[:50]}")
    except Exception as e:
        errors.append(f"Itinerary JSON parse error: {str(e)[:100]}")
        validated_days = []

    ms = int((time.time() - start) * 1000)
    timings = dict(state.get("node_timings", {}))
    timings["itinerary_node"] = ms

    tracker.log_span("itinerary_node", {"tour": raw.get('name', '')[:50]},
                     {"days_parsed": len(validated_days)},
                     tokens=tokens_used, latency_ms=ms)

    return {**state, "itinerary_days": validated_days,
            "errors": errors, "node_timings": timings}


logger.info("✓ Nodes 1-3 defined")

In [ ]:
# ── Tool 2: SQL Generator ──────────────────────────────────────────────────────

def escape_sql(value: Optional[str]) -> str:
    """Safely escape a value for SQL INSERT."""
    if value is None:
        return 'NULL'
    value = str(value).replace("'", "''")
    return f"'{value}'"


def generate_sql_node(state: TourProcessingState) -> TourProcessingState:
    """Tool: Generate SQL INSERT statements from processed data."""
    start = time.time()
    raw = state["raw_tour"]
    rewritten = state.get("rewritten_content") or {}
    days = state.get("itinerary_days", [])
    idx = state["tour_index"]

    # IDs (would come from actual DB sequences in production)
    area_id = 1
    country_id = 1
    destination_id = idx + 1
    provider_id = 1
    tour_id = idx + 1

    # Infer destination from tour name/subtitle
    subtitle = raw.get('subtitle', '') or ''
    locations = re.findall(r'[A-Z][A-Z\s]+(?:PRABANG|VIENTIANE|PAKSE|VIENG|PLATEAU|ISLANDS|LAOS|CHAMPASAK|PHONSAVAN)',
                           subtitle.upper())
    dest_name = locations[0].title() if locations else raw.get('name', 'Laos')[:50]

    # Provider SQL
    provider_name = raw.get('provider_normalized', 'Tiger Trail')
    provider_sql = f"""
-- Provider
INSERT IGNORE INTO providers (id, name, created_at) 
VALUES ({provider_id}, {escape_sql(provider_name)}, NOW());"""

    # Destination SQL
    destination_sql = f"""
-- Destination  
INSERT INTO destinations (id, country_id, name, status, created_at)
VALUES ({destination_id}, {country_id}, {escape_sql(dest_name)}, 'ACTIVE', NOW())
ON DUPLICATE KEY UPDATE name=VALUES(name);"""

    # Tour SQL
    title = escape_sql(rewritten.get('title') or raw.get('name', 'Tour'))
    duration = escape_sql(rewritten.get('duration') or raw.get('duration'))
    price = raw.get('price_parsed')
    price_sql = str(price) if price else 'NULL'
    sku = escape_sql(rewritten.get('sku') or raw.get('sku'))
    summary = escape_sql(rewritten.get('summary', ''))
    highlight = escape_sql(rewritten.get('highlight', ''))
    included = escape_sql(rewritten.get('included', ''))
    not_included = escape_sql(rewritten.get('not_included', ''))

    tour_sql = f"""
-- Tour: {raw.get('name', '')[:60]}
INSERT INTO tours (
    id, destination_id, sku, title, duration, price,
    summary, hightlight, included, not_included, status, created_at
) VALUES (
    {tour_id}, {destination_id}, {sku}, {title}, {duration}, {price_sql},
    {summary}, {highlight}, {included}, {not_included}, 'ACTIVE', NOW()
);"""

    # Itinerary SQLs
    itinerary_sqls = []
    for i, day in enumerate(days, start=1):
        itin_id = (tour_id - 1) * 20 + i  # Simple ID generation
        itin_sql = f"""
-- Day {day['day_number']}: {day['title'][:50]}
INSERT INTO itineraries (id, tour_id, title, level, description, status, created_at)
VALUES (
    {itin_id}, {tour_id},
    {escape_sql(day['title'])},
    {escape_sql(day.get('level', 'moderate'))},
    {escape_sql(day['description'])},
    'ACTIVE', NOW()
);"""
        itinerary_sqls.append(itin_sql)

    # Guardrail: validate generated SQL
    validation_errors = []
    for sql in [tour_sql] + itinerary_sqls:
        if 'DROP' in sql.upper() or 'DELETE' in sql.upper():
            validation_errors.append("SQL injection detected")
        if sql.count("'") % 2 != 0:
            validation_errors.append("Unbalanced quotes in SQL")

    sql_output = SQLOutput(
        provider_sql=provider_sql,
        destination_sql=destination_sql,
        tour_sql=tour_sql,
        itinerary_sqls=itinerary_sqls,
        tour_name=raw.get('name', ''),
        validation_passed=len(validation_errors) == 0,
        validation_errors=validation_errors,
    )

    ms = int((time.time() - start) * 1000)
    timings = dict(state.get("node_timings", {}))
    timings["sql_node"] = ms

    tracker.log_span("generate_sql_node",
                     {"tour": raw.get('name', '')[:50]},
                     {"itinerary_count": len(itinerary_sqls),
                      "validation_passed": sql_output.validation_passed},
                     latency_ms=ms)

    return {**state, "sql_output": sql_output.model_dump(),
            "validation_passed": sql_output.validation_passed,
            "node_timings": timings}


# ── Node 5: Validation Gate ────────────────────────────────────────────────────

def validate_node(state: TourProcessingState) -> TourProcessingState:
    """Final quality check — LLM-as-judge for content quality."""
    start = time.time()
    rewritten = state.get("rewritten_content") or {}
    errors = list(state.get("errors", []))

    # Rule-based checks
    issues = []
    title = rewritten.get('title', '')
    summary = rewritten.get('summary', '')

    if len(title) < 10:
        issues.append("Title too short")
    if len(summary) < 50:
        issues.append("Summary too short")
    if not rewritten.get('highlight'):
        issues.append("No highlights")
    if not rewritten.get('included'):
        issues.append("No inclusions")

    # Check SQL output
    sql_out = state.get("sql_output") or {}
    if sql_out.get('validation_errors'):
        issues.extend(sql_out['validation_errors'])

    validation_passed = len(issues) == 0

    ms = int((time.time() - start) * 1000)
    timings = dict(state.get("node_timings", {}))
    timings["validate_node"] = ms

    if not validation_passed:
        errors.extend([f"VALIDATION: {i}" for i in issues])

    tracker.log_span("validate_node",
                     {"tour": rewritten.get('title', '')[:50]},
                     {"passed": validation_passed, "issues": issues},
                     latency_ms=ms)

    return {**state, "validation_passed": validation_passed,
            "errors": errors, "node_timings": timings}


# ── Routing Function ───────────────────────────────────────────────────────────

def route_after_validate(state: TourProcessingState) -> str:
    """Route: pass → end, fail with retries left → retry rewrite."""
    if state["validation_passed"]:
        return "end"
    if state.get("retry_count", 0) < 2:
        logger.warning(f"Retrying rewrite (attempt {state.get('retry_count', 0) + 1})")
        return "retry"
    return "end"  # Give up after 2 retries


logger.info("✓ Nodes 4-5 and routing defined")

In [ ]:
# ── Build the LangGraph ───────────────────────────────────────────────────────

def increment_retry(state: TourProcessingState) -> TourProcessingState:
    return {**state, "retry_count": state.get("retry_count", 0) + 1,
            "rewritten_content": None}  # Reset for retry


builder = StateGraph(TourProcessingState)

# Add nodes
builder.add_node("parse", parse_node)
builder.add_node("rewrite", rewrite_node)
builder.add_node("itinerary", itinerary_node)
builder.add_node("generate_sql", generate_sql_node)
builder.add_node("validate", validate_node)
builder.add_node("increment_retry", increment_retry)

# Edges
builder.add_edge(START, "parse")
builder.add_edge("parse", "rewrite")
builder.add_edge("rewrite", "itinerary")
builder.add_edge("itinerary", "generate_sql")
builder.add_edge("generate_sql", "validate")
builder.add_conditional_edges(
    "validate",
    route_after_validate,
    {"end": END, "retry": "increment_retry"}
)
builder.add_edge("increment_retry", "rewrite")  # Retry loop

# Compile with memory checkpointer
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

logger.info("✓ LangGraph compiled successfully")
logger.info("Graph nodes: parse → rewrite → itinerary → generate_sql → validate")
logger.info("Retry path: validate → increment_retry → rewrite (max 2 retries)")

## Criterion 1: Problem & System Design — Run the Pipeline

In [ ]:
# ── Run Pipeline on a Sample Tour ────────────────────────────────────────────

def process_tour(tour: RawTourProduct, idx: int) -> dict:
    """Process a single tour through the full agent pipeline."""
    config = {"configurable": {"thread_id": f"tour-{idx}-{tour.name[:20]}"}}

    initial_state: TourProcessingState = {
        "raw_tour": tour.model_dump(),
        "tour_index": idx,
        "rewritten_content": None,
        "itinerary_days": [],
        "sql_output": None,
        "validation_passed": False,
        "retry_count": 0,
        "errors": [],
        "warnings": [],
        "trace_id": f"trace-{idx}-{int(time.time())}",
        "node_timings": {},
    }

    result = graph.invoke(initial_state, config=config)
    return result


# Process first 3 tours as demo
print("=" * 70)
print("DEMO: Processing 3 tours from Tiger Trail Excel")
print("=" * 70)

demo_tours = raw_tours[:3]
results = []

for i, tour in enumerate(demo_tours):
    print(f"\n[{i+1}/3] Processing: {tour.name[:60]}")
    result = process_tour(tour, i)
    results.append(result)

    sql_out = result.get("sql_output") or {}
    rewritten = result.get("rewritten_content") or {}
    print(f"  ✓ Title: {rewritten.get('title', 'N/A')[:60]}")
    print(f"  ✓ SQL generated: {len(sql_out.get('itinerary_sqls', []))} itinerary days")
    print(f"  ✓ Validation: {'PASSED' if result.get('validation_passed') else 'FAILED'}")
    if result.get('errors'):
        print(f"  ! Errors: {result['errors'][:2]}")

print(f"\n✓ Pipeline complete for {len(results)} tours")

## Criterion 6: Evaluation & Testing — Quality Metrics

In [ ]:
# ── Evaluation Framework ───────────────────────────────────────────────────────

EVAL_DATASET = [
    {
        "tour_name": "VIENTIANE STOPOVER",
        "expected_keywords_in_summary": ["Vientiane", "capital"],
        "expected_keywords_in_title": ["Vientiane"],
        "min_itinerary_days": 1,
        "required_inclusions": ["guide", "transport"],
        "required_exclusions": ["flight"],
    },
    {
        "tour_name": "NORTHERN LAOS",
        "expected_keywords_in_summary": ["Luang Prabang"],
        "expected_keywords_in_title": ["Luang"],
        "min_itinerary_days": 3,
        "required_inclusions": ["guide"],
        "required_exclusions": ["visa"],
    },
]


def evaluate_result(result: dict, eval_case: dict) -> dict:
    """Evaluate a processed tour against expected criteria."""
    rewritten = result.get("rewritten_content") or {}
    days = result.get("itinerary_days", [])
    sql = result.get("sql_output") or {}

    scores = {}

    # 1. Title quality
    title = rewritten.get('title', '').lower()
    title_keywords_hit = sum(
        1 for kw in eval_case.get('expected_keywords_in_title', [])
        if kw.lower() in title
    )
    scores['title_relevance'] = title_keywords_hit / max(1, len(eval_case.get('expected_keywords_in_title', [1])))

    # 2. Summary quality
    summary = rewritten.get('summary', '').lower()
    summary_keywords_hit = sum(
        1 for kw in eval_case.get('expected_keywords_in_summary', [])
        if kw.lower() in summary
    )
    scores['summary_relevance'] = summary_keywords_hit / max(1, len(eval_case.get('expected_keywords_in_summary', [1])))

    # 3. Itinerary completeness
    min_days = eval_case.get('min_itinerary_days', 0)
    scores['itinerary_completeness'] = 1.0 if len(days) >= min_days else len(days) / max(1, min_days)

    # 4. Inclusions coverage
    included = rewritten.get('included', '').lower()
    incl_hit = sum(1 for kw in eval_case.get('required_inclusions', [])
                   if kw.lower() in included)
    scores['inclusions_coverage'] = incl_hit / max(1, len(eval_case.get('required_inclusions', [1])))

    # 5. SQL validity
    scores['sql_valid'] = 1.0 if sql.get('validation_passed') else 0.0

    # 6. Content length quality
    scores['content_richness'] = min(1.0, len(rewritten.get('summary', '')) / 200)

    overall = sum(scores.values()) / len(scores)
    return {
        "tour": eval_case['tour_name'],
        "scores": scores,
        "overall": round(overall, 3),
        "passed": overall >= 0.7,
    }


# Run evaluation
print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)

eval_results = []
for i, (result, eval_case) in enumerate(zip(results[:2], EVAL_DATASET)):
    if result:
        score = evaluate_result(result, eval_case)
        eval_results.append(score)
        status = "✓ PASS" if score['passed'] else "✗ FAIL"
        print(f"\n{status} Tour: {score['tour']}")
        print(f"  Overall: {score['overall']:.1%}")
        for metric, val in score['scores'].items():
            bar = '█' * int(val * 10) + '░' * (10 - int(val * 10))
            print(f"  {metric:<25} {bar} {val:.0%}")

if eval_results:
    avg_overall = sum(r['overall'] for r in eval_results) / len(eval_results)
    print(f"\nAverage Score: {avg_overall:.1%}")
    print(f"Pass Rate: {sum(1 for r in eval_results if r['passed'])}/{len(eval_results)}")

## Full Output — SQL Statements

In [ ]:
# ── Generate Complete SQL Output File ─────────────────────────────────────────

def generate_full_sql(results: list[dict], output_path: str = "/mnt/user-data/outputs/tours_insert.sql"):
    """Write all SQL INSERT statements to a file."""
    lines = [
        "-- ============================================================",
        f"-- Tour Content SQL — Generated by Agentic AI System",
        f"-- Source: Tiger Trail Products Excel",
        f"-- Generated: {datetime.utcnow().isoformat()} UTC",
        f"-- Tours processed: {len(results)}",
        "-- ============================================================",
        "",
        "SET FOREIGN_KEY_CHECKS = 0;",
        "",
        "-- ===== AREA =====",
        "INSERT IGNORE INTO areas (id, name, created_at) VALUES (1, 'Southeast Asia', NOW());",
        "",
        "-- ===== COUNTRY =====",
        "INSERT IGNORE INTO countries (id, area_id, name, subtitle, status, created_at)",
        "VALUES (1, 1, 'Laos', 'Land of a Million Elephants', 'ACTIVE', NOW());",
        "",
    ]

    seen_providers = set()
    for result in results:
        sql_out = result.get("sql_output") or {}
        if not sql_out:
            continue

        tour_name = sql_out.get('tour_name', 'Unknown')
        lines.append(f"\n-- ===== TOUR: {tour_name[:60].upper()} =====")

        if sql_out.get('provider_sql') and 'Tiger Trail' not in seen_providers:
            lines.append(sql_out['provider_sql'])
            seen_providers.add('Tiger Trail')

        if sql_out.get('destination_sql'):
            lines.append(sql_out['destination_sql'])

        lines.append(sql_out.get('tour_sql', ''))

        for itin_sql in sql_out.get('itinerary_sqls', []):
            lines.append(itin_sql)

    lines.extend([
        "",
        "SET FOREIGN_KEY_CHECKS = 1;",
        f"-- End of file — {len(results)} tours inserted",
    ])

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    logger.info(f"✓ SQL written to {output_path}")
    return output_path


sql_path = generate_full_sql(results)
print(f"\n✓ SQL output: {sql_path}")
print("\nFirst 50 lines preview:")
with open(sql_path) as f:
    preview = f.readlines()[:50]
print(''.join(preview))

## Criterion 9: Production Readiness — Batch Processing

In [ ]:
# ── Batch Processing: all tours with progress tracking ───────────────────────

def process_all_tours(
    tours: list[RawTourProduct],
    batch_size: int = 5,
    max_tours: int = 10,
) -> list[dict]:
    """Production-grade batch processor with error isolation."""
    all_results = []
    failed = 0
    tours_to_process = tours[:max_tours]

    print(f"Processing {len(tours_to_process)} tours (batch_size={batch_size})...")

    for i in range(0, len(tours_to_process), batch_size):
        batch = tours_to_process[i:i + batch_size]
        print(f"\nBatch {i // batch_size + 1}: tours {i+1}-{i+len(batch)}")

        for j, tour in enumerate(batch):
            idx = i + j
            try:
                result = process_tour(tour, idx)
                all_results.append(result)
                status = "✓" if result.get('validation_passed') else "⚠"
                print(f"  {status} [{idx+1}] {tour.name[:50]}")
            except Exception as e:
                failed += 1
                logger.error(f"Tour {idx} failed: {e}")
                all_results.append({"raw_tour": {"name": tour.name}, "sql_output": None,
                                     "errors": [str(e)], "validation_passed": False})

    print(f"\n{'='*60}")
    print(f"Processed: {len(all_results)} | Passed: {sum(1 for r in all_results if r.get('validation_passed'))} | Failed: {failed}")
    return all_results


# Process all tours (limited for demo)
all_results = process_all_tours(raw_tours, batch_size=5, max_tours=10)

# Generate full SQL
final_sql_path = generate_full_sql(all_results, "/mnt/user-data/outputs/tours_insert_full.sql")
print(f"\n✓ Full SQL: {final_sql_path}")

## Criterion 10: Demo & Summary

In [ ]:
# ── Final Summary Dashboard ────────────────────────────────────────────────────

obs = tracker.summary()
cache_stats = cache.stats()

passed_count = sum(1 for r in all_results if r.get('validation_passed'))
total_count = len(all_results)
tours_with_sql = sum(1 for r in all_results if r.get('sql_output'))
total_itinerary_days = sum(
    len(r.get('sql_output', {}).get('itinerary_sqls', []))
    for r in all_results if r.get('sql_output')
)

print("""
╔══════════════════════════════════════════════════════════════════════╗
║          TOUR CONTENT REWRITER — AGENTIC AI SYSTEM SUMMARY         ║
╠══════════════════════════════════════════════════════════════════════╣
║  PROCESSING RESULTS                                                  ║""")
print(f"║  Tours processed       : {total_count:<43}║")
print(f"║  Validation passed     : {passed_count}/{total_count} ({passed_count/max(1,total_count)*100:.0f}%){' '*(37-len(str(passed_count)+str(total_count)))}║")
print(f"║  SQL INSERT sets       : {tours_with_sql:<43}║")
print(f"║  Itinerary days        : {total_itinerary_days:<43}║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  PERFORMANCE & COST                                              ║")
print(f"║  Total LLM tokens      : {obs['total_tokens']:<43}║")
print(f"║  Estimated cost        : ${obs['estimated_cost_usd']:<42}║")
print(f"║  Cache hit rate        : {cache_stats['hit_rate']:<43}║")
print(f"║  Pipeline latency      : {obs['total_latency_ms']}ms{' '*(40-len(str(obs['total_latency_ms'])))}║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  SYSTEM ARCHITECTURE                                             ║")
print("║  Framework: LangGraph (StateGraph + MemorySaver)                ║")
print("║  Nodes: parse → rewrite → itinerary → generate_sql → validate  ║")
print("║  Guardrails: Pydantic validation + SQL injection protection      ║")
print("║  RAG: DB schema as knowledge base + field-level retrieval        ║")
print("║  Observability: ObservabilityTracker (Langfuse-ready)            ║")
print("║  Caching: MD5-keyed file cache for LLM responses                 ║")
print("║  Retry: Up to 2 retries on validation failure                    ║")
print("╚══════════════════════════════════════════════════════════════════╝")

print("\n📄 Output files:")
print(f"  - /mnt/user-data/outputs/tours_insert_full.sql")